In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (CICERON)

This notebook curates multiple toxicity-related peptide subsets from **CICERON** starting from the original CSV export. CICERON provides a functional annotation field (`Function`) and a class label (`Class`). Here we normalize sequence formatting, filter peptides into activity-specific subsets based on `Function`, perform duplicate consistency checks within each subset, and export standardized datasets and metadata for downstream analysis.

- **Toxic effect / endpoint:** hemolytic, cytotoxic, embryotoxic, toxic
- **Source:** CICERON
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset

The pipeline performs the following steps:
- **Loads the CICERON CSV file** and standardizes key fields:
  - renames `Sequence` → `sequence` and `Class` → `label`,
  - normalizes `Function` (strip + lowercase),
  - cleans sequences by removing artifacts like `~` and trimming whitespace.
- **Derives toxicity-specific subsets** by filtering the `Function` annotation:
  - **hemolytic**: `Function` contains “haemolytic/hemolytic”
  - **toxic**: `Function` equals “toxic”
  - **cytotoxic**: `Function` contains “cytotoxic”
  - **embryotoxic**: `Function` contains “embryotoxic”
  Each subset keeps `(sequence, label)` where `label` is the original CICERON class.
- **Checks duplicates by sequence within each subset**:
  - unique sequences are preserved,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors and exported for review.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated datasets** for each subset and a consolidated error table.

In [2]:
name_source = "CICERON"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_ciceron = pd.read_csv(f"{PATH_INPUT}/{name_source}/CICERON.csv")

In [4]:
df_ciceron = (
    df_ciceron
    .rename(columns={"Sequence": "sequence", "Class": "label"})
    .assign(
        Function=lambda d: d["Function"].str.strip().str.lower(),
        sequence=lambda d: d["sequence"].str.replace("~", "", regex=False).str.strip()
    )
)

- Separate dataset by toxic effects

In [5]:
df_hemolytic = (
    df_ciceron
    .loc[
        df_ciceron["Function"].str.contains(r"ha?emolytic", case=False, na=False),
        ["sequence", "label"]
    ]
    .reset_index(drop=True)
)

df_hemolytic.shape

(88, 2)

In [6]:
df_toxic = (
    df_ciceron
    .loc[
        df_ciceron["Function"].str.lower().str.strip() == "toxic",
        ["sequence", "label"]
    ]
    .reset_index(drop=True)
)
df_toxic.shape

(9, 2)

In [7]:
df_cytotoxic = (
    df_ciceron
    .loc[
        df_ciceron["Function"].str.contains("cytotoxic", case=False, na=False),
        ["sequence", "label"]
    ]
    .reset_index(drop=True)
)
df_cytotoxic.shape

(7, 2)

In [8]:
df_embryotoxic = (
    df_ciceron
    .loc[
        df_ciceron["Function"].str.contains("embryotoxic", case=False, na=False),
        ["sequence", "label"]
    ]
    .reset_index(drop=True)
)
df_embryotoxic.shape

(3, 2)

- Checking duplicates

In [9]:
df_remove_duplicated_hemolytic, df_errors_hemolytic, df_unique_hemolytic = processing_duplicated(df_hemolytic, group_seq="sequence", sort_key="label")

In [10]:
df_remove_duplicated_cytotoxic, df_errors_cytotoxic, df_unique_cytotoxic = processing_duplicated(df_cytotoxic, group_seq="sequence", sort_key="label")

In [11]:
df_remove_duplicated_toxic, df_errors_toxic, df_unique_toxic = processing_duplicated(df_toxic, group_seq="sequence", sort_key="label")

In [12]:
df_remove_duplicated_embryo, df_errors_embryo, df_unique_embryo = processing_duplicated(df_embryotoxic, group_seq="sequence", sort_key="label")

In [13]:
df_full_hemolytic = pd.concat([df_unique_hemolytic, df_remove_duplicated_hemolytic])
df_full_cytotoxic = pd.concat([df_unique_cytotoxic, df_remove_duplicated_cytotoxic])
df_full_toxic = pd.concat([df_unique_toxic, df_remove_duplicated_toxic])
df_full_embryo = pd.concat([df_remove_duplicated_embryo, df_unique_embryo])
df_full = pd.concat([df_full_hemolytic, df_full_cytotoxic, df_full_toxic, df_full_embryo])
df_errors = pd.concat([df_errors_hemolytic, df_errors_cytotoxic, df_errors_toxic, df_errors_embryo])

In [14]:
df_errors.shape

(0, 1)

- Working with metada

In [15]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [16]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_ciceron)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2024,
 'last update date': datetime.datetime(2024, 4, 30, 0, 0),
 'download date': Timestamp('2025-09-01 00:00:00'),
 'file format': 'csv',
 'peptide property': 'ACE inhibitor, antioxidant, multilabel, dipeptidyl peptidase IV inhibitor, antihypertensive, antimicrobial, antibacterial, celiac toxic, opioid, immunomodulating, antithrombotic, anticancer, neuropeptides, alpha-glucosidase inhibitor, PEP-inhibitory, hemolytic, toxic, renin inhibitor, alpha-amylase inhibitor, dipeptidyl peptidase III inhibitor, binding peptides, antiamnestic, antifungal, heparin binding, γ-glutamyl, CaMKII Inhibitor, antiviral, cholestrol-lowering, HMG-CoA reductase inhibitor, Ileum contracting, antiinflammatory, mineral-binding, anxiolitic-like, CaMPDE inhibitor, cyclooxygenase-1 inhibitor, blood-brain barrier peptides, membrane -active , mitogenic, AChE inhibitor, vasorelaxant, vasoconstrictor, embry

- Exporting data

In [17]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [18]:
df_full_hemolytic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_embryo.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_embryotoxic_dataset.csv", index=False)
df_full_cytotoxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_cytotoxic_dataset.csv", index=False)
df_full_toxic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)

df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)